In [1]:
!uname -a && cat /etc/*release
!pwd
!ls -la

Linux 19ec6f001b1c 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
/content
total 16
drwxr-xr-x 1 root root 4096 Feb  6 14:31 .
drwxr-xr-x 1 root root 4096 Mar 19 10:50 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data


In [2]:
!nvidia-smi

Thu Mar 19 10:54:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>

#define TILE_SIZE 32
// Definiamo quanti elementi calcolerà ogni thread (4)
#define COARSEN_FACTOR 4

// Kernel Ottimizzato: Shared Memory Tiling + Thread Coarsening
__global__ void matMulCoarsened(const float *A, const float *B, float *C, int n) {
    __shared__ float s_A[TILE_SIZE][TILE_SIZE];
    __shared__ float s_B[TILE_SIZE][TILE_SIZE];

    // Array nei registri locali: qui accumuliamo i 4 risultati!
    float sum[COARSEN_FACTOR] = {0.0f};

    // Un blocco ora ha dimensioni (8, 32) -> 256 thread totali
    int tx = threadIdx.x; // da 0 a 7
    int ty = threadIdx.y; // da 0 a 31

    int row = blockIdx.y * TILE_SIZE + ty;
    int col_start = blockIdx.x * TILE_SIZE + tx * COARSEN_FACTOR;

    // Indice lineare del thread per la fase di caricamento cooperativo
    int tid = ty * (TILE_SIZE / COARSEN_FACTOR) + tx;

    for (int t = 0; t < (n + TILE_SIZE - 1) / TILE_SIZE; ++t) {

        // --- 1. Caricamento Cooperativo ---
        // 256 thread devono riempire 1024 elementi. Ognuno fa 4 letture.
        #pragma unroll
        for (int load_offset = 0; load_offset < (TILE_SIZE * TILE_SIZE); load_offset += 256) {
            int load_tid = tid + load_offset;
            int tile_row = load_tid / TILE_SIZE;
            int tile_col = load_tid % TILE_SIZE;

            int global_a_row = blockIdx.y * TILE_SIZE + tile_row;
            int global_a_col = t * TILE_SIZE + tile_col;
            int global_b_row = t * TILE_SIZE + tile_row;
            int global_b_col = blockIdx.x * TILE_SIZE + tile_col;

            if (global_a_row < n && global_a_col < n)
                s_A[tile_row][tile_col] = A[global_a_row * n + global_a_col];
            else
                s_A[tile_row][tile_col] = 0.0f;

            if (global_b_row < n && global_b_col < n)
                s_B[tile_row][tile_col] = B[global_b_row * n + global_b_col];
            else
                s_B[tile_row][tile_col] = 0.0f;
        }

        __syncthreads();

        // --- 2. Calcolo a Manetta sui Registri ---
        #pragma unroll
        for (int k = 0; k < TILE_SIZE; ++k) {
            // IL TRUCCO: Leggiamo A una volta sola e lo teniamo in un registro
            float valA = s_A[ty][k];

            // Ricicliamo valA per fare 4 moltiplicazioni diverse!
            #pragma unroll
            for (int c = 0; c < COARSEN_FACTOR; ++c) {
                sum[c] += valA * s_B[k][tx * COARSEN_FACTOR + c];
            }
        }
        __syncthreads();
    }

    // --- 3. Scrittura Finale ---
    #pragma unroll
    for (int c = 0; c < COARSEN_FACTOR; ++c) {
        int final_col = col_start + c;
        if (row < n && final_col < n) {
            C[row * n + final_col] = sum[c];
        }
    }
}

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    size_t bytes = (size_t)n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // ATTENZIONE: Il blocco ora ha meno thread sull'asse X
    dim3 threadsPerBlock(TILE_SIZE / COARSEN_FACTOR, TILE_SIZE);
    dim3 numBlocks((n + TILE_SIZE - 1) / TILE_SIZE, (n + TILE_SIZE - 1) / TILE_SIZE);

    clock_t start = clock();

    matMulCoarsened<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, n);

    cudaDeviceSynchronize();
    clock_t end = clock();

    printf("Execution Time (Thread Coarsening FP32): %.4f seconds\\n", (double)(end - start) / CLOCKS_PER_SEC);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''

with open('cuda_matrixmult_tiled.cu', 'w') as f:
    f.write(cell_str)

import subprocess

# Compilazione
subprocess.run(["nvcc", "-O3", "cuda_matrixmult_tiled.cu", "-o", "cuda_matrixmult_tiled"])

# Profilazione con nvprof
subprocess.run(["nvprof", "./cuda_matrixmult_tiled", "20000"])

CompletedProcess(args=['nvprof', './cuda_matrixmult_tiled', '20000'], returncode=0)

In [4]:
!nvcc -arch=sm_75 -O3 cuda_matrixmult_tiled.cu -o cuda_matrixmult_tiled
!nvprof ./cuda_matrixmult_tiled 5000

==1129== NVPROF is profiling process 1129, command: ./cuda_matrixmult_tiled 5000
Execution Time (Thread Coarsening FP32): 0.2685 seconds
==1129== Profiling application: ./cuda_matrixmult_tiled 5000
==1129== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   80.79%  268.23ms         1  268.23ms  268.23ms  268.23ms  matMulCoarsened(float const *, float const *, float*, int)
                   12.58%  41.755ms         2  20.877ms  20.829ms  20.926ms  [CUDA memcpy HtoD]
                    6.63%  22.014ms         1  22.014ms  22.014ms  22.014ms  [CUDA memcpy DtoH]
      API calls:   47.38%  268.25ms         1  268.25ms  268.25ms  268.25ms  cudaDeviceSynchronize
                   40.47%  229.10ms         3  76.368ms  74.491us  228.94ms  cudaMalloc
                   11.40%  64.565ms         3  21.522ms  21.045ms  22.382ms  cudaMemcpy
                    0.44%  2.4681ms         3  822.71us  287.91us  1.1335ms  cudaFree
    

In [5]:
!nvprof ./cuda_matrixmult_tiled 10000

==1149== NVPROF is profiling process 1149, command: ./cuda_matrixmult_tiled 10000
Execution Time (Thread Coarsening FP32): 1.6981 seconds
==1149== Profiling application: ./cuda_matrixmult_tiled 10000
==1149== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   86.10%  1.74829s         1  1.74829s  1.74829s  1.74829s  matMulCoarsened(float const *, float const *, float*, int)
                    9.45%  191.89ms         2  95.945ms  92.240ms  99.650ms  [CUDA memcpy HtoD]
                    4.45%  90.390ms         1  90.390ms  90.390ms  90.390ms  [CUDA memcpy DtoH]
      API calls:   80.30%  1.74830s         1  1.74830s  1.74830s  1.74830s  cudaDeviceSynchronize
                   13.00%  283.14ms         3  94.379ms  90.761ms  99.884ms  cudaMemcpy
                    6.42%  139.81ms         3  46.605ms  120.98us  139.47ms  cudaMalloc
                    0.22%  4.7813ms         3  1.5938ms  392.45us  2.3008ms  cudaFree
  

In [6]:
!nvprof ./cuda_matrixmult_tiled 15000

==1173== NVPROF is profiling process 1173, command: ./cuda_matrixmult_tiled 15000
Execution Time (Thread Coarsening FP32): 6.0904 seconds
==1173== Profiling application: ./cuda_matrixmult_tiled 15000
==1173== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   91.27%  6.17389s         1  6.17389s  6.17389s  6.17389s  matMulCoarsened(float const *, float const *, float*, int)
                    5.67%  383.79ms         2  191.89ms  191.87ms  191.92ms  [CUDA memcpy HtoD]
                    3.05%  206.61ms         1  206.61ms  206.61ms  206.61ms  [CUDA memcpy DtoH]
      API calls:   89.48%  6.17391s         1  6.17391s  6.17391s  6.17391s  cudaDeviceSynchronize
                    8.57%  591.23ms         3  197.08ms  192.12ms  206.95ms  cudaMemcpy
                    1.86%  128.27ms         3  42.756ms  110.33us  128.04ms  cudaMalloc
                    0.08%  5.2030ms         3  1.7343ms  602.30us  2.4913ms  cudaFree
  

In [7]:
!nvprof ./cuda_matrixmult_tiled 20000

==1220== NVPROF is profiling process 1220, command: ./cuda_matrixmult_tiled 20000
Execution Time (Thread Coarsening FP32): 14.4982 seconds
==1220== Profiling application: ./cuda_matrixmult_tiled 20000
==1220== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   92.86%  14.5805s         1  14.5805s  14.5805s  14.5805s  matMulCoarsened(float const *, float const *, float*, int)
                    4.79%  752.54ms         2  376.27ms  359.47ms  393.07ms  [CUDA memcpy HtoD]
                    2.34%  368.08ms         1  368.08ms  368.08ms  368.08ms  [CUDA memcpy DtoH]
      API calls:   91.56%  14.5806s         1  14.5806s  14.5806s  14.5806s  cudaDeviceSynchronize
                    7.04%  1.12153s         3  373.84ms  359.70ms  393.31ms  cudaMemcpy
                    1.34%  213.64ms         3  71.215ms  191.99us  213.26ms  cudaMalloc
                    0.04%  5.7786ms         3  1.9262ms  887.72us  2.7920ms  cudaFree
 